# TRF+ Client Series Construction

This notebook creates simple, inspectable D+1 prediction series from the cleaned TRF+ dataframe.

Keep only two final objects in mind:

- `series_rules`: priority-ordered rules that assign cleaned TRF+ rows to prediction series.
- `daily_series`: wide daily target table with one column per prediction series.

Tune the force-select lists, company split overrides, and residual rules, then re-run from that point.

## 0. Imports And Config

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import re
import sys
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    import yaml
except ImportError:
    yaml = None

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)


@dataclass
class SeriesConfig:
    input_path: str = "data/temp/output_clean.csv"
    ruleset_path: str = "configs/rulesets/loreal_cash_in_v1.yaml"
    date_col: str = "VALUE_DATE"
    amount_col: str = "SIGNED_AMOUNT"
    movement_col: str = "CASH_MOVEMENT_TYPE_SHORTNAME"
    movement_value: str = "TRF+"
    id_col: str = "ID"
    company_text_candidates: tuple[str, ...] = ("LABEL", "DESCRIPTION", "CPTY_NAME", "CPTY_SHORTNAME")
    company_col: str = "company_name"

    # Intervals are [min_abs_amount, max_abs_amount).
    amount_bins: tuple[tuple[float, float, str], ...] = (
        (0.0, 20_000.0, "low"),
        (20_000.0, 100_000.0, "medium"),
        (100_000.0, 1_000_000.0, "high"),
        (1_000_000.0, np.inf, "huge"),
    )

    max_selected_companies: int = 30
    min_company_rows: int = 5
    min_company_active_days: int = 3
    min_company_total_abs: float = 100_000.0

    # A selected company is split only when at least this many bins are meaningful.
    min_split_rows: int = 5
    min_split_active_days: int = 3
    min_split_abs_share: float = 0.05
    min_meaningful_company_bins: int = 2

    residual_low_max: float = 20_000.0
    residual_medium_max: float = 100_000.0
    daily_freq: str = "D"
    n_company_plots: int = 8
    output_dir: str = "data/temp/client_series"


cfg = SeriesConfig()

# Manual tuning knobs. Use normalized company names as displayed in `company_features`.
FORCE_SELECT_COMPANIES: list[str] = []
FORCE_EXCLUDE_COMPANIES: list[str] = []

# Leave empty to plot the top `cfg.n_company_plots` selected companies by score.
COMPANIES_TO_PLOT: list[str] = []

asdict(cfg)

## 1. Helper Functions

In [ ]:
def normalize_text(value) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    text = text.upper()
    text = "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )
    text = re.sub(r"[^A-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def normalize_company(name) -> str | None:
    name = normalize_text(name)
    if not name:
        return None

    legal_forms = [
        r"\bS\s*A\s*U\b", r"\bS\s*A\b", r"\bS\s*L\s*U\b", r"\bS\s*L\b",
        r"\bB\s*V\b", r"\bN\s*V\b", r"\bS\s*A\s*R\s*L\b", r"\bS\s*C\s*A\b",
        r"\bC\s*B\b", r"\bCO\s*LTD\b", r"\bCO\s*LIMITED\b", r"\bLTD\b", r"\bLIMITED\b",
    ]
    for pattern in legal_forms:
        name = re.sub(pattern, " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    if not name or re.fullmatch(r"\d+", name):
        return None
    if re.fullmatch(r"\d+", name[:9].strip()):
        return None
    if name.startswith("INCOMING"):
        return None
    return name or None


def extract_company(text) -> str | None:
    if pd.isna(text):
        return None
    text = str(text).strip()
    if not text:
        return None
    patterns = [
        r"TRANSFERENCIA DE\s+([^,]+)",
        r"/PT/FT/BO/([^/]+)",
        r"/BO/([^/]+)",
        r"^([^/]+)/",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            candidate = match.group(1).strip()
            if candidate and not re.fullmatch(r"\d+", candidate):
                return candidate
    return None


def sanitize_label(value: str, max_len: int = 80) -> str:
    text = normalize_text(value) or "UNKNOWN"
    text = re.sub(r"[^A-Z0-9]+", "_", text).strip("_")
    return (text or "UNKNOWN")[:max_len]


def amount_bin_label(abs_amount: float, cfg: SeriesConfig = cfg) -> str:
    if pd.isna(abs_amount):
        return "unknown"
    x = float(abs_amount)
    for low, high, label in cfg.amount_bins:
        if x >= low and x < high:
            return label
    return "unknown"


def amount_bin_bounds(label: str, cfg: SeriesConfig = cfg) -> tuple[float, float]:
    for low, high, bin_label in cfg.amount_bins:
        if label == bin_label:
            return float(low), float(high)
    raise ValueError(f"Unknown amount bin: {label}")


def parse_amount_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    return pd.to_numeric(
        series.astype("string")
        .str.replace("\u00A0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def first_non_empty_text(df: pd.DataFrame, columns: tuple[str, ...]) -> pd.Series:
    out = pd.Series(pd.NA, index=df.index, dtype="string")
    for col in columns:
        if col not in df.columns:
            continue
        candidate = df[col].astype("string").str.strip().mask(lambda s: s.eq(""), pd.NA)
        out = out.fillna(candidate)
    return out


def normalized_entropy(values) -> float:
    counts = pd.Series(values).value_counts(dropna=True)
    if len(counts) <= 1:
        return 0.0
    p = counts / counts.sum()
    return float(-(p * np.log(p)).sum() / np.log(len(counts)))


def top_share(values) -> float:
    counts = pd.Series(values).value_counts(dropna=True)
    if counts.sum() == 0:
        return np.nan
    return float(counts.iloc[0] / counts.sum())


def safe_cv2(values) -> float:
    arr = pd.Series(values).dropna().astype(float)
    if len(arr) <= 1:
        return np.nan
    mean = arr.mean()
    if abs(mean) < 1e-12:
        return np.nan
    return float((arr.std(ddof=1) / mean) ** 2)


def gap_stats(dates) -> dict[str, float]:
    dates = pd.Series(pd.to_datetime(dates).dropna().unique()).sort_values()
    if len(dates) <= 1:
        return {
            "median_gap_days": np.nan,
            "mean_gap_days": np.nan,
            "cv_gap_days": np.nan,
            "min_gap_days": np.nan,
            "max_gap_days": np.nan,
        }
    gaps = dates.diff().dropna().dt.days.astype(float)
    mean_gap = gaps.mean()
    cv_gap = gaps.std(ddof=1) / mean_gap if mean_gap and not np.isnan(mean_gap) else np.nan
    return {
        "median_gap_days": float(gaps.median()),
        "mean_gap_days": float(mean_gap),
        "cv_gap_days": float(cv_gap) if not np.isnan(cv_gap) else np.nan,
        "min_gap_days": float(gaps.min()),
        "max_gap_days": float(gaps.max()),
    }

In [ ]:
def load_ruleset_payload(path: str | Path) -> dict:
    if yaml is None:
        return {}
    ruleset_path = Path(path)
    if not ruleset_path.is_absolute():
        ruleset_path = ROOT / ruleset_path
    if not ruleset_path.exists():
        return {}
    with ruleset_path.open("r", encoding="utf-8") as handle:
        return yaml.safe_load(handle) or {}


def apply_ruleset_filters_light(df: pd.DataFrame, ruleset_payload: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    filters = ruleset_payload.get("filters", {}) if ruleset_payload else {}
    current = df.copy()
    rows = []

    for column, rule in filters.items():
        before = len(current)
        if column not in current.columns:
            rows.append({"rule": column, "status": "missing_column", "before": before, "removed": 0, "after": before})
            continue

        case_sensitive = bool(rule.get("case_sensitive", False))
        normalized = current[column].astype("string")
        if not case_sensitive:
            normalized = normalized.str.lower()
        keep = pd.Series(True, index=current.index)

        include_values = rule.get("include_values") or []
        if include_values:
            values = {str(v if case_sensitive else str(v).lower()) for v in include_values}
            keep &= normalized.isin(values)

        exclude_values = rule.get("exclude_values") or []
        if exclude_values:
            values = {str(v if case_sensitive else str(v).lower()) for v in exclude_values}
            keep &= ~normalized.isin(values)

        exclude_contains = rule.get("exclude_contains") or []
        if exclude_contains:
            values = [str(v if case_sensitive else str(v).lower()) for v in exclude_contains]
            pattern = "|".join(re.escape(v) for v in values)
            keep &= ~normalized.fillna("").str.contains(pattern, regex=True, na=False)

        current = current.loc[keep].copy()
        after = len(current)
        rows.append({"rule": column, "status": "applied", "before": before, "removed": before - after, "after": after})

    return current.reset_index(drop=True), pd.DataFrame(rows)


def canonicalize_company_names(tx: pd.DataFrame, threshold: int = 90) -> pd.DataFrame:
    out = tx.copy()
    original = out["company_name"].copy()
    names = original.dropna().value_counts()

    try:
        from rapidfuzz import fuzz

        def score(a, b):
            return fuzz.token_sort_ratio(a, b)
    except ImportError:
        from difflib import SequenceMatcher

        def score(a, b):
            return 100 * SequenceMatcher(None, a, b).ratio()

    mapping = {}
    used = set()
    ordered_names = names.index.tolist()
    for name in ordered_names:
        if name in used:
            continue
        group = [name]
        used.add(name)
        for other in ordered_names:
            if other in used:
                continue
            if score(name, other) >= threshold:
                group.append(other)
                used.add(other)
        best = max(group, key=lambda n: names.loc[n])
        for item in group:
            mapping[item] = best

    manual_mapping = {
        "COFARES SDAD COOPERATIVA FARMACEUT": "COFARES SOCIEDAD COOPERATIVA FARMA",
        "SANTANDER FACTORING Y CONFIRMING E F C": "SANTANDER FACTORING Y CONFIRMING S",
        "AYESA IBERMATICAS A U": "AYESA IBERMATICA",
        "PAYPAL EUROPE R L ET CIE S C A": "PAYPAL EUROPE R L ET CIE",
    }
    mapping.update(manual_mapping)

    out["company_name"] = original.map(mapping).fillna(original)
    out["company_found"] = out["company_name"].notna()
    out["company_key"] = out["company_name"].fillna("NO_COMPANY")
    return out

## 2. Load And Clean TRF+ Rows

This keeps the existing `data/temp/output_clean.csv` input. The YAML ruleset is applied again as a safety check before focusing on TRF+.

In [ ]:
input_path = ROOT / cfg.input_path
df = pd.read_csv(input_path)
print("Raw input:", df.shape)

df.columns = [str(c).strip() for c in df.columns]
ruleset_payload = load_ruleset_payload(cfg.ruleset_path)
df_rules, rule_waterfall = apply_ruleset_filters_light(df, ruleset_payload)
display(rule_waterfall)
print("After YAML ruleset safety filters:", df_rules.shape)

df_trf = df_rules.loc[df_rules[cfg.movement_col].astype("string").eq(cfg.movement_value)].copy()
print(f"After {cfg.movement_value} filter:", df_trf.shape)

for col in ["CAPTURE_DATE", "ISSUE_DATE", "TRADE_DATE", "VALUE_DATE", "MATCH_DATE"]:
    if col in df_trf.columns:
        df_trf[col] = pd.to_datetime(df_trf[col], errors="coerce").dt.normalize()

for col in ["AMOUNT", "SIGNED_AMOUNT", "ORIGIN_AMOUNT"]:
    if col in df_trf.columns:
        df_trf[col] = parse_amount_series(df_trf[col])

if cfg.id_col in df_trf.columns:
    before = len(df_trf)
    df_trf = df_trf.drop_duplicates(subset=cfg.id_col, keep="first").copy()
    print(f"Dropped duplicated {cfg.id_col}: {before - len(df_trf)}")

company_text = first_non_empty_text(df_trf, cfg.company_text_candidates)
extracted_company = company_text.map(extract_company).map(normalize_company)
if cfg.company_col in df_trf.columns:
    existing_company = df_trf[cfg.company_col].map(normalize_company)
else:
    existing_company = pd.Series(pd.NA, index=df_trf.index, dtype="object")

df_trf["company_name_raw"] = existing_company.fillna(extracted_company)
df_trf["company_name"] = df_trf["company_name_raw"].map(normalize_company)
df_trf = canonicalize_company_names(df_trf)

df_trf["amount"] = pd.to_numeric(df_trf[cfg.amount_col], errors="coerce")
df_trf["abs_amount"] = df_trf["amount"].abs()
df_trf["amount_bin"] = df_trf["abs_amount"].map(lambda x: amount_bin_label(x, cfg))
df_trf["dayofweek"] = df_trf[cfg.date_col].dt.dayofweek
df_trf["day_name"] = df_trf[cfg.date_col].dt.day_name()
df_trf["dayofmonth"] = df_trf[cfg.date_col].dt.day
df_trf = df_trf.dropna(subset=[cfg.date_col, "amount"]).copy()

print("Final cleaned TRF+ rows:", df_trf.shape)
print("Date range:", df_trf[cfg.date_col].min(), "->", df_trf[cfg.date_col].max())
print("Known company rate:", round(float(df_trf["company_found"].mean()), 3))
print("Unique known companies:", df_trf.loc[df_trf["company_found"], "company_name"].nunique())

display(
    df_trf.groupby("amount_bin", dropna=False)
    .agg(rows=("amount", "size"), total_amount=("amount", "sum"), total_abs_amount=("abs_amount", "sum"), companies=("company_name", "nunique"))
    .reset_index()
)
display(df_trf.head())

## 3. Top Companies

Decide which companies deserve dedicated prediction series. Use `FORCE_SELECT_COMPANIES` and `FORCE_EXCLUDE_COMPANIES` when business context beats the automatic score.

In [ ]:
def classify_company_behavior(row) -> str:
    if not bool(row.get("company_found", False)):
        return "unknown_company"

    n_rows = row["n_rows"]
    active_days = row["n_active_days"]
    total_abs = row["total_abs_amount"]
    cv2 = row["cv2_daily_abs_amount"]
    median_gap = row["median_gap_days"]
    cv_gap = row["cv_gap_days"]
    dow_top = row["dow_top_share"]
    dom_top = row["dom_top_share"]
    spike_ratio = row["max_to_median_abs"]

    if active_days >= 8 and (
        (not pd.isna(median_gap) and 6 <= median_gap <= 8 and (pd.isna(cv_gap) or cv_gap <= 0.45))
        or (not pd.isna(dow_top) and dow_top >= 0.60)
    ):
        return "periodic_weekly"
    if active_days >= 4 and (
        (not pd.isna(median_gap) and 25 <= median_gap <= 35 and (pd.isna(cv_gap) or cv_gap <= 0.35))
        or (not pd.isna(dom_top) and dom_top >= 0.60)
    ):
        return "periodic_monthly"
    if n_rows <= 3 and total_abs >= cfg.residual_medium_max:
        return "rare_large"
    if active_days >= 30:
        return "frequent_stable" if not pd.isna(cv2) and cv2 <= 1.0 else "frequent_erratic"
    if active_days >= 6:
        if not pd.isna(cv2) and cv2 <= 0.49:
            return "intermittent_stable"
        if not pd.isna(spike_ratio) and spike_ratio >= 10:
            return "lumpy_spiky"
        return "intermittent_lumpy"
    return "sparse_tail"


def build_company_features(tx: pd.DataFrame, cfg: SeriesConfig) -> pd.DataFrame:
    known = tx[tx["company_found"]].copy()
    if known.empty:
        return pd.DataFrame()

    date_col = cfg.date_col
    end_date = tx[date_col].max()
    base = (
        known.groupby("company_name", dropna=False)
        .agg(
            n_rows=("amount", "size"),
            n_active_days=(date_col, "nunique"),
            total_amount=("amount", "sum"),
            total_abs_amount=("abs_amount", "sum"),
            mean_abs_amount=("abs_amount", "mean"),
            median_abs_amount=("abs_amount", "median"),
            p90_abs_amount=("abs_amount", lambda s: s.quantile(0.90)),
            max_abs_amount=("abs_amount", "max"),
            first_date=(date_col, "min"),
            last_date=(date_col, "max"),
            dow_entropy=("dayofweek", normalized_entropy),
            dow_top_share=("dayofweek", top_share),
            dom_entropy=("dayofmonth", normalized_entropy),
            dom_top_share=("dayofmonth", top_share),
            company_found=("company_found", "first"),
        )
        .reset_index()
    )

    daily = (
        known.groupby(["company_name", date_col], dropna=False)
        .agg(daily_amount=("amount", "sum"), daily_abs_amount=("abs_amount", "sum"))
        .reset_index()
    )
    daily_feat = (
        daily.groupby("company_name", dropna=False)
        .agg(
            mean_daily_abs_amount=("daily_abs_amount", "mean"),
            median_daily_abs_amount=("daily_abs_amount", "median"),
            max_daily_abs_amount=("daily_abs_amount", "max"),
            cv2_daily_abs_amount=("daily_abs_amount", safe_cv2),
        )
        .reset_index()
    )
    base = base.merge(daily_feat, on="company_name", how="left")

    gap_rows = []
    for company, group in daily.groupby("company_name", dropna=False):
        gap_rows.append({"company_name": company, **gap_stats(group[date_col])})
    gaps_expanded = pd.DataFrame(gap_rows)
    base = base.merge(gaps_expanded, on="company_name", how="left")

    base["days_since_last"] = (end_date - base["last_date"]).dt.days
    base["span_days"] = (base["last_date"] - base["first_date"]).dt.days + 1
    base["max_to_median_abs"] = base["max_abs_amount"] / base["median_abs_amount"].replace(0, np.nan)
    base["behavior_group"] = base.apply(classify_company_behavior, axis=1)

    total_abs_all = base["total_abs_amount"].sum()
    base["share_total_abs"] = base["total_abs_amount"] / total_abs_all if total_abs_all else np.nan
    base["rank_amount_pct"] = base["total_abs_amount"].rank(pct=True)
    base["rank_rows_pct"] = base["n_rows"].rank(pct=True)
    base["rank_active_days_pct"] = base["n_active_days"].rank(pct=True)
    max_days = max(float(base["days_since_last"].max()), 1.0)
    base["recency_score"] = 1.0 - (base["days_since_last"].clip(lower=0) / max_days)
    base["regularity_bonus"] = base["behavior_group"].isin(["periodic_weekly", "periodic_monthly", "frequent_stable"]).astype(float)
    base["company_score"] = (
        0.45 * base["rank_amount_pct"].fillna(0)
        + 0.25 * base["rank_rows_pct"].fillna(0)
        + 0.15 * base["rank_active_days_pct"].fillna(0)
        + 0.10 * base["recency_score"].fillna(0)
        + 0.05 * base["regularity_bonus"].fillna(0)
    )

    eligible = (
        (base["n_rows"] >= cfg.min_company_rows)
        & (base["n_active_days"] >= cfg.min_company_active_days)
        & (base["total_abs_amount"] >= cfg.min_company_total_abs)
    )
    base["auto_selected"] = False
    auto_idx = base.loc[eligible].sort_values(["company_score", "total_abs_amount", "n_rows"], ascending=False).head(cfg.max_selected_companies).index
    base.loc[auto_idx, "auto_selected"] = True

    forced_select = {normalize_company(x) for x in FORCE_SELECT_COMPANIES}
    forced_select.discard(None)
    forced_exclude = {normalize_company(x) for x in FORCE_EXCLUDE_COMPANIES}
    forced_exclude.discard(None)

    base["is_selected_company"] = base["auto_selected"] | base["company_name"].isin(forced_select)
    base.loc[base["company_name"].isin(forced_exclude), "is_selected_company"] = False
    base["selection_reason"] = np.select(
        [base["company_name"].isin(forced_select), base["company_name"].isin(forced_exclude), base["auto_selected"]],
        ["force_selected", "force_excluded", "auto_score"],
        default="not_selected",
    )
    return base.sort_values(["is_selected_company", "company_score", "total_abs_amount"], ascending=[False, False, False]).reset_index(drop=True)


company_features = build_company_features(df_trf, cfg)
selected_companies = company_features.loc[company_features["is_selected_company"], "company_name"].tolist()

print("Selected companies:", len(selected_companies))
display(
    company_features.loc[:, [
        "company_name", "is_selected_company", "selection_reason", "company_score",
        "n_rows", "n_active_days", "total_abs_amount", "share_total_abs",
        "behavior_group", "days_since_last", "dow_top_share", "median_gap_days",
    ]].head(80)
)

In [ ]:
def plot_company_selection(company_features: pd.DataFrame) -> go.Figure:
    f = company_features.copy()
    f["selection_status"] = np.where(f["is_selected_company"], "selected", "not_selected")
    fig = px.scatter(
        f,
        x="n_rows",
        y="total_abs_amount",
        color="selection_status",
        symbol="behavior_group",
        size="share_total_abs",
        hover_name="company_name",
        hover_data={
            "company_score": ":.3f",
            "n_active_days": True,
            "total_abs_amount": ":,.0f",
            "share_total_abs": ":.2%",
            "behavior_group": True,
            "days_since_last": True,
        },
        log_x=True,
        log_y=True,
        title="Company selection landscape",
        labels={"n_rows": "Payments", "total_abs_amount": "Total absolute amount"},
    )
    fig.update_layout(template="plotly_white", height=620, legend_title_text="")
    fig.update_traces(marker={"opacity": 0.72, "line": {"width": 0.5, "color": "white"}})
    return fig


fig_company_selection = plot_company_selection(company_features)
fig_company_selection.show()

## 4. Suggested Series For Selected Companies

The notebook proposes company-specific amount splits. Edit `company_series_overrides` directly if a split is too granular or not granular enough.

Rules are interval based: `min_abs_amount <= abs(SIGNED_AMOUNT) < max_abs_amount`.

In [ ]:
def suggest_company_series_rules(tx: pd.DataFrame, selected_companies: list[str], cfg: SeriesConfig) -> pd.DataFrame:
    rows = []
    for company in selected_companies:
        sdf = tx.loc[tx["company_name"].eq(company)].copy()
        if sdf.empty:
            continue
        by_bin = (
            sdf.groupby("amount_bin", dropna=False)
            .agg(n_rows=("amount", "size"), n_active_days=(cfg.date_col, "nunique"), total_abs_amount=("abs_amount", "sum"))
            .reset_index()
        )
        total_abs = by_bin["total_abs_amount"].sum()
        by_bin["share_company_abs"] = by_bin["total_abs_amount"] / total_abs if total_abs else 0.0
        by_bin["meaningful_bin"] = (
            (by_bin["n_rows"] >= cfg.min_split_rows)
            & (by_bin["n_active_days"] >= cfg.min_split_active_days)
            & (by_bin["share_company_abs"] >= cfg.min_split_abs_share)
        )
        meaningful = by_bin.loc[by_bin["meaningful_bin"], "amount_bin"].tolist()
        should_split = len(meaningful) >= cfg.min_meaningful_company_bins

        if should_split:
            for amount_bin in [label for _, _, label in cfg.amount_bins if label in by_bin["amount_bin"].tolist()]:
                low, high = amount_bin_bounds(amount_bin, cfg)
                stats = by_bin.loc[by_bin["amount_bin"].eq(amount_bin)].iloc[0]
                rows.append({
                    "enabled": True,
                    "company_name": company,
                    "split_label": amount_bin,
                    "min_abs_amount": low,
                    "max_abs_amount": high,
                    "series_name": f"company__{sanitize_label(company)}__{amount_bin}",
                    "comment": (
                        f"selected company split; rows={int(stats['n_rows'])}; "
                        f"active_days={int(stats['n_active_days'])}; share={stats['share_company_abs']:.1%}"
                    ),
                })
        else:
            rows.append({
                "enabled": True,
                "company_name": company,
                "split_label": "all",
                "min_abs_amount": 0.0,
                "max_abs_amount": np.inf,
                "series_name": f"company__{sanitize_label(company)}__all",
                "comment": "selected company kept as one series; amount bins did not create several meaningful subseries",
            })
    return pd.DataFrame(rows)


company_series_suggestions = suggest_company_series_rules(df_trf, selected_companies, cfg)

# Edit this dataframe if needed, then re-run downstream cells.
company_series_overrides = company_series_suggestions.copy()
display(company_series_overrides)

In [ ]:
def build_company_rules(company_series_overrides: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "priority", "enabled", "series_name", "rule_type", "company_name", "company_found",
        "min_abs_amount", "max_abs_amount", "behavior_group", "comment",
    ]
    if company_series_overrides.empty:
        return pd.DataFrame(columns=columns)
    rules = company_series_overrides.loc[company_series_overrides["enabled"].fillna(True).astype(bool)].copy()
    rules["priority"] = range(1000, 1000 + len(rules))
    rules["rule_type"] = "selected_company"
    rules["company_found"] = "*"
    rules["behavior_group"] = rules.get("split_label", "")
    return rules.loc[:, columns]


def assign_series_from_rules(tx: pd.DataFrame, series_rules: pd.DataFrame) -> pd.DataFrame:
    assigned = tx.copy()
    assigned["series_name"] = pd.NA
    assigned["matched_rule_type"] = pd.NA
    assigned["matched_rule_priority"] = pd.NA
    active_rules = series_rules.loc[series_rules["enabled"].fillna(True).astype(bool)].sort_values("priority")

    for _, rule in active_rules.iterrows():
        mask = assigned["series_name"].isna()
        company_name = rule.get("company_name", "*")
        if pd.notna(company_name) and str(company_name) != "*":
            mask &= assigned["company_name"].eq(company_name)

        company_found = rule.get("company_found", "*")
        if pd.notna(company_found) and str(company_found) != "*":
            mask &= assigned["company_found"].eq(bool(company_found))

        min_amount = float(rule.get("min_abs_amount", 0.0) or 0.0)
        max_amount = rule.get("max_abs_amount", np.inf)
        max_amount = np.inf if pd.isna(max_amount) else float(max_amount)
        mask &= assigned["abs_amount"].ge(min_amount) & assigned["abs_amount"].lt(max_amount)

        assigned.loc[mask, "series_name"] = rule["series_name"]
        assigned.loc[mask, "matched_rule_type"] = rule["rule_type"]
        assigned.loc[mask, "matched_rule_priority"] = rule["priority"]
    return assigned


def plot_selected_company_series(tx: pd.DataFrame, company_rules: pd.DataFrame, company_name: str) -> go.Figure:
    company_rules = company_rules.loc[company_rules["company_name"].eq(company_name)].copy()
    if company_rules.empty:
        raise ValueError(f"No company rules found for {company_name}")
    provisional = assign_series_from_rules(tx.loc[tx["company_name"].eq(company_name)].copy(), company_rules)
    daily = provisional.groupby([cfg.date_col, "series_name"], dropna=False)["amount"].sum().reset_index()
    fig = px.line(
        daily,
        x=cfg.date_col,
        y="amount",
        color="series_name",
        markers=True,
        title=f"Selected-company series check: {company_name}",
        labels={"amount": "Daily amount", cfg.date_col: "Value date"},
    )
    fig.update_layout(template="plotly_white", height=430, legend_title_text="")
    return fig


company_rules_preview = build_company_rules(company_series_overrides)
companies_to_plot = COMPANIES_TO_PLOT or selected_companies[: cfg.n_company_plots]
for company in companies_to_plot:
    fig = plot_selected_company_series(df_trf, company_rules_preview, company)
    fig.show()

## 5. Residual Rules For Remaining Rows

These rules apply after selected-company rules. Keep them broad, then split only when the plots show a real pattern worth modeling separately.

In [ ]:
residual_rule_templates = pd.DataFrame([
    {
        "enabled": True,
        "series_name": "residual__unknown_company__low_weekly",
        "rule_type": "residual",
        "company_name": "*",
        "company_found": False,
        "min_abs_amount": 0.0,
        "max_abs_amount": cfg.residual_low_max,
        "behavior_group": "unknown_company_low_weekly",
        "comment": "unknown-company low amounts; inspect for pooled weekly seasonality",
    },
    {
        "enabled": True,
        "series_name": "residual__known_small_company__low_weekly",
        "rule_type": "residual",
        "company_name": "*",
        "company_found": True,
        "min_abs_amount": 0.0,
        "max_abs_amount": cfg.residual_low_max,
        "behavior_group": "known_low_weekly",
        "comment": "non-selected known companies with low amounts; often useful as a pooled weekly series",
    },
    {
        "enabled": True,
        "series_name": "residual__unknown_company__medium",
        "rule_type": "residual",
        "company_name": "*",
        "company_found": False,
        "min_abs_amount": cfg.residual_low_max,
        "max_abs_amount": cfg.residual_medium_max,
        "behavior_group": "unknown_company_medium",
        "comment": "unknown-company medium amounts",
    },
    {
        "enabled": True,
        "series_name": "residual__known_company__medium_intermittent",
        "rule_type": "residual",
        "company_name": "*",
        "company_found": True,
        "min_abs_amount": cfg.residual_low_max,
        "max_abs_amount": cfg.residual_medium_max,
        "behavior_group": "medium_intermittent",
        "comment": "non-selected known companies with medium amounts",
    },
    {
        "enabled": True,
        "series_name": "residual__unknown_company__rare_large",
        "rule_type": "residual",
        "company_name": "*",
        "company_found": False,
        "min_abs_amount": cfg.residual_medium_max,
        "max_abs_amount": np.inf,
        "behavior_group": "unknown_company_rare_large",
        "comment": "unknown-company large peaks; likely hard to predict but important to show honestly",
    },
    {
        "enabled": True,
        "series_name": "residual__known_company__rare_large",
        "rule_type": "residual",
        "company_name": "*",
        "company_found": True,
        "min_abs_amount": cfg.residual_medium_max,
        "max_abs_amount": np.inf,
        "behavior_group": "known_rare_large",
        "comment": "non-selected known-company large peaks; likely sparse/intermittent",
    },
    {
        "enabled": True,
        "series_name": "residual__catch_all",
        "rule_type": "residual",
        "company_name": "*",
        "company_found": "*",
        "min_abs_amount": 0.0,
        "max_abs_amount": np.inf,
        "behavior_group": "catch_all",
        "comment": "safety net; should be tiny or zero after the rules above",
    },
])

# Edit this table if the residual plots suggest a simpler or more granular split.
residual_rules = residual_rule_templates.copy()
residual_rules["priority"] = range(9000, 9000 + len(residual_rules))
residual_rules = residual_rules.loc[:, [
    "priority", "enabled", "series_name", "rule_type", "company_name", "company_found",
    "min_abs_amount", "max_abs_amount", "behavior_group", "comment",
]]
display(residual_rules)

## 6. Build Final `series_rules` And `daily_series`

In [ ]:
company_rules = build_company_rules(company_series_overrides)
series_rules = pd.concat([company_rules, residual_rules], ignore_index=True).sort_values("priority").reset_index(drop=True)

assigned_transactions = assign_series_from_rules(df_trf, series_rules)

calendar = pd.date_range(df_trf[cfg.date_col].min(), df_trf[cfg.date_col].max(), freq=cfg.daily_freq, name=cfg.date_col)
series_names = series_rules.loc[series_rules["enabled"].fillna(True).astype(bool), "series_name"].drop_duplicates().tolist()

series_daily_long = (
    assigned_transactions.groupby([cfg.date_col, "series_name"], dropna=False)
    .agg(amount=("amount", "sum"), rows=("amount", "size"))
    .reset_index()
)
full_index = pd.MultiIndex.from_product([calendar, series_names], names=[cfg.date_col, "series_name"])
daily_series = (
    series_daily_long.set_index([cfg.date_col, "series_name"])["amount"]
    .reindex(full_index, fill_value=0.0)
    .unstack("series_name")
    .sort_index()
)

series_review = (
    assigned_transactions.groupby(["series_name", "matched_rule_type"], dropna=False)
    .agg(
        rows=("amount", "size"),
        active_days=(cfg.date_col, "nunique"),
        total_amount=("amount", "sum"),
        total_abs_amount=("abs_amount", "sum"),
        companies=("company_key", "nunique"),
        first_date=(cfg.date_col, "min"),
        last_date=(cfg.date_col, "max"),
    )
    .reset_index()
    .sort_values("total_abs_amount", ascending=False)
)
series_review["share_total_abs"] = series_review["total_abs_amount"] / series_review["total_abs_amount"].sum()

print("series_rules:", series_rules.shape)
print("daily_series:", daily_series.shape)
display(series_rules)
display(series_review)
display(daily_series.head())

In [ ]:
def validate_outputs(tx: pd.DataFrame, assigned: pd.DataFrame, daily_series: pd.DataFrame, selected_companies: list[str]) -> dict[str, float | int]:
    if len(tx) != len(assigned):
        raise AssertionError(f"Row count changed: tx={len(tx)}, assigned={len(assigned)}")
    if assigned["series_name"].isna().any():
        display(assigned.loc[assigned["series_name"].isna()].head(20))
        raise AssertionError(f"Unassigned rows: {assigned['series_name'].isna().sum()}")

    selected_mask = assigned["company_name"].isin(selected_companies)
    bad_selected = selected_mask & ~assigned["matched_rule_type"].eq("selected_company")
    if bad_selected.any():
        display(assigned.loc[bad_selected, [cfg.date_col, "company_name", "amount", "abs_amount", "series_name", "matched_rule_type"]].head(20))
        raise AssertionError("Some selected-company rows were not assigned by selected-company rules")

    bad_remaining = (~selected_mask) & ~assigned["matched_rule_type"].eq("residual")
    if bad_remaining.any():
        display(assigned.loc[bad_remaining, [cfg.date_col, "company_name", "amount", "abs_amount", "series_name", "matched_rule_type"]].head(20))
        raise AssertionError("Some remaining rows were not assigned by residual rules")

    raw_daily = tx.groupby(cfg.date_col)["amount"].sum().reindex(daily_series.index, fill_value=0.0)
    recon_daily = daily_series.sum(axis=1).reindex(daily_series.index, fill_value=0.0)
    max_abs_reconstruction_error = float((raw_daily - recon_daily).abs().max())
    if max_abs_reconstruction_error > 1e-6:
        raise AssertionError(f"Daily reconstruction error is too large: {max_abs_reconstruction_error}")

    return {
        "rows": int(len(assigned)),
        "series": int(daily_series.shape[1]),
        "selected_companies": int(len(selected_companies)),
        "max_abs_reconstruction_error": max_abs_reconstruction_error,
    }


validation_summary = validate_outputs(df_trf, assigned_transactions, daily_series, selected_companies)
validation_summary

## 7. Plotly Checks

In [ ]:
def plot_residual_overview(series_review: pd.DataFrame) -> go.Figure:
    f = series_review.loc[series_review["matched_rule_type"].eq("residual")].copy()
    fig = px.scatter(
        f,
        x="rows",
        y="total_abs_amount",
        size="active_days",
        color="series_name",
        hover_name="series_name",
        hover_data={"active_days": True, "companies": True, "share_total_abs": ":.2%", "total_abs_amount": ":,.0f"},
        log_x=True,
        log_y=True,
        title="Residual series overview",
        labels={"rows": "Rows", "total_abs_amount": "Total absolute amount"},
    )
    fig.update_layout(template="plotly_white", height=540, legend_title_text="")
    return fig


def plot_weekly_seasonality(assigned: pd.DataFrame, contains: str = "low_weekly") -> go.Figure:
    f = assigned.loc[assigned["series_name"].astype("string").str.contains(contains, regex=False, na=False)].copy()
    if f.empty:
        return go.Figure().update_layout(title=f"No rows found for weekly check: {contains}", template="plotly_white")
    weekly = (
        f.groupby(["series_name", "dayofweek", "day_name"], dropna=False)
        .agg(mean_daily_amount=("amount", "mean"), total_amount=("amount", "sum"), rows=("amount", "size"))
        .reset_index()
        .sort_values("dayofweek")
    )
    fig = px.bar(
        weekly,
        x="day_name",
        y="total_amount",
        color="series_name",
        barmode="group",
        hover_data={"rows": True, "mean_daily_amount": ":,.0f"},
        title="Weekly profile for low residual series",
        labels={"day_name": "Weekday", "total_amount": "Total amount"},
    )
    fig.update_layout(template="plotly_white", height=460, legend_title_text="")
    return fig


def plot_rare_peaks(assigned: pd.DataFrame, contains: str = "rare_large") -> go.Figure:
    f = assigned.loc[assigned["series_name"].astype("string").str.contains(contains, regex=False, na=False)].copy()
    if f.empty:
        return go.Figure().update_layout(title=f"No rows found for rare peak check: {contains}", template="plotly_white")
    fig = px.scatter(
        f,
        x=cfg.date_col,
        y="amount",
        color="series_name",
        size="abs_amount",
        hover_data={"company_name": True, "abs_amount": ":,.0f", "amount_bin": True},
        title="Rare large residual peaks",
        labels={"amount": "Movement amount", cfg.date_col: "Value date"},
    )
    fig.update_layout(template="plotly_white", height=520, legend_title_text="")
    return fig


def plot_reconstruction(tx: pd.DataFrame, daily_series: pd.DataFrame) -> go.Figure:
    raw_daily = tx.groupby(cfg.date_col)["amount"].sum().reindex(daily_series.index, fill_value=0.0)
    recon_daily = daily_series.sum(axis=1)
    plot_df = pd.DataFrame({
        cfg.date_col: daily_series.index,
        "original_daily_trf": raw_daily.values,
        "sum_created_series": recon_daily.values,
    })
    long = plot_df.melt(id_vars=cfg.date_col, value_vars=["original_daily_trf", "sum_created_series"], var_name="series", value_name="amount")
    fig = px.line(
        long,
        x=cfg.date_col,
        y="amount",
        color="series",
        title="Reconstruction check: original TRF+ total vs created series total",
        labels={"amount": "Daily amount", cfg.date_col: "Value date"},
    )
    fig.update_layout(template="plotly_white", height=460, legend_title_text="")
    return fig


def plot_top_daily_series(daily_series: pd.DataFrame, series_review: pd.DataFrame, top_n: int = 12) -> go.Figure:
    top_series = series_review.sort_values("total_abs_amount", ascending=False)["series_name"].head(top_n).tolist()
    plot_df = daily_series[top_series].reset_index().melt(id_vars=cfg.date_col, var_name="series_name", value_name="amount")
    fig = px.line(
        plot_df,
        x=cfg.date_col,
        y="amount",
        color="series_name",
        title=f"Top {top_n} created prediction series",
        labels={"amount": "Daily amount", cfg.date_col: "Value date"},
    )
    fig.update_layout(template="plotly_white", height=620, legend_title_text="")
    return fig


fig_residual_overview = plot_residual_overview(series_review)
fig_weekly = plot_weekly_seasonality(assigned_transactions)
fig_rare_peaks = plot_rare_peaks(assigned_transactions)
fig_reconstruction = plot_reconstruction(df_trf, daily_series)
fig_top_series = plot_top_daily_series(daily_series, series_review)

for fig in [fig_residual_overview, fig_weekly, fig_rare_peaks, fig_reconstruction, fig_top_series]:
    assert isinstance(fig, go.Figure)
    fig.show()

## 8. Optional Export

Exports are explicit on purpose. Turn `SAVE_OUTPUTS` on only when the rules look good.

In [ ]:
SAVE_OUTPUTS = False

if SAVE_OUTPUTS:
    outdir = ROOT / cfg.output_dir
    outdir.mkdir(parents=True, exist_ok=True)

    rules_to_save = series_rules.copy()
    rules_to_save["max_abs_amount"] = rules_to_save["max_abs_amount"].replace(np.inf, "inf")
    rules_to_save.to_csv(outdir / "series_rules.csv", index=False)
    daily_series.to_parquet(outdir / "daily_series.parquet")
    assigned_transactions.to_parquet(outdir / "assigned_transactions.parquet", index=False)

    if yaml is not None:
        with (outdir / "series_rules.yaml").open("w", encoding="utf-8") as handle:
            yaml.safe_dump(rules_to_save.to_dict(orient="records"), handle, sort_keys=False, allow_unicode=True)

    print(f"Saved series outputs to {outdir.resolve()}")